In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import mysql.connector
import time
import re
import csv

# Initialize WebDriver
def get_driver():
    options = Options()
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    options.add_argument("--headless")  # Run in headless mode
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Scroll and Load More Cars
def scroll_page(driver):
    scroll_attempts = 0
    last_height = driver.execute_script("return document.body.scrollHeight")

    while scroll_attempts < 10:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)  # Wait for new elements to load
        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            scroll_attempts += 1
        else:
            scroll_attempts = 0
            last_height = new_height

    print("Scrolling completed.")

# Extract Car Data
def extract_car_data(driver, location, car_type):
    car_list = []

    # Wait for elements to load
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "ClsBox"))
    )

    car_elements = driver.find_elements(By.CLASS_NAME, "cardColumn")

    for element in car_elements:
        try:
            # Extract Car Name
            name_element = element.find_elements(By.XPATH, ".//h3[contains(@class, 'title')]")
            name = name_element[0].text.strip() if name_element else "N/A"

            # Extract Price
            price_element = element.find_elements(By.XPATH, ".//div[contains(@class, 'Price')]/p")
            price = price_element[0].text.strip() if price_element else "N/A"
            price = extract_price(price)

            # Extract Details (KM Driven, Fuel Type, Transmission)
            details_element = element.find_elements(By.CLASS_NAME, "dotsDetails")
            details_text = details_element[0].get_attribute("innerText").strip() if details_element else ""
            details_list = [item.strip() for item in details_text.split("•")] if details_text else []

            kms = details_list[0].replace(" kms", "").replace(",", "").strip() if len(details_list) > 0 else "0"
            fuel_type = details_list[1].strip() if len(details_list) > 1 else "Unknown"
            transmission_type = details_list[2].strip() if len(details_list) > 2 else "Unknown"

            # Convert KM Driven to Integer
            try:
                kms = int(kms)
            except ValueError:
                kms = 0

            # Extract Location
            location_element = element.find_elements(By.XPATH, ".//div[contains(@class, 'distanceText')]")
            full_location = location_element[0].text.strip() if location_element else location
            full_location = full_location.split(",")[-1].strip()

            model = name.split()[0]

            # Extract Car Details Page Link
            car_link_element = element.find_elements(By.TAG_NAME, "a")
            car_link = car_link_element[0].get_attribute("href") if car_link_element else "N/A"

            # Extract Mileage from Car Details Page
            mileage = extract_mileage(driver, car_link) if car_link != "N/A" else "N/A"

            # Store Data
            car_list.append({
                "Name": name,
                "Price": price,
                "Kilometers_Driven": kms,
                "Fuel_Type": fuel_type,
                "Transmission_Type": transmission_type,
                "Location": full_location,
                "Car_Type": car_type,
                "Model": model,
                "Mileage": mileage,
                "URL": car_link
            })

        except Exception as e:
            print(f"Error extracting data: {e}")

    return car_list

# Extract Mileage from Car Details Page
# Extract Mileage from Car Details Page
def extract_mileage(driver, car_url):
    driver.execute_script("window.open('', '_blank');")  # Open new tab
    driver.switch_to.window(driver.window_handles[-1])  # Switch to new tab
    driver.get(car_url)
    time.sleep(2)  # Wait for page to load

    try:
        # Locate the mileage based on its label
        mileage_element = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//div[@class='label' and text()='Mileage ']/following-sibling::span"))
        )
        mileage = mileage_element.text.strip()
    except Exception:
        mileage = "N/A"

    driver.close()  # Close the details tab
    driver.switch_to.window(driver.window_handles[0])  # Switch back to main page

    return mileage


# Extract Price
def extract_price(price_str):
    match = re.search(r"([\d.]+)\s*Lakh", price_str)
    if match:
        return int(float(match.group(1)) * 100000)  # Convert Lakh to integer
    match = re.search(r"([\d,]+)", price_str)
    if match:
        return int(match.group(1).replace(",", ""))  # Convert to integer
    return None  # Return None if no valid price found

# Save Data to MySQL
def save_to_mysql(data, host, user, password, database):
    try:
        conn = mysql.connector.connect(
            host=host, user=user, password=password, database=database
        )
        cursor = conn.cursor()

        query = """
        INSERT INTO dummy_car (name, price, kilometers_driven, fuel_type, transmission_type, location, car_type, model, mileage, url)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        for car in data:
            values = (
                car["Name"], car["Price"], car["Kilometers_Driven"], car["Fuel_Type"],
                car["Transmission_Type"], car["Location"], car["Car_Type"], car["Model"],
                car["Mileage"], car["URL"]
            )
            cursor.execute(query, values)

        conn.commit()
        print(f"{len(data)} records inserted into MySQL!")

    except mysql.connector.Error as err:
        print(f"MySQL Error: {err}")

    finally:
        cursor.close()
        conn.close()

# Save Data to CSV
def save_to_csv(data, filename="car_details.csv"):
    with open(filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)
    print(f"Data saved to {filename}")

# Main Function
def main(url, location, car_type, host, user, password, database):
    print(f"Fetching data from: {url}")

    driver = get_driver()
    driver.get(url)
    time.sleep(2)

    scroll_page(driver)

    car_data = extract_car_data(driver, location, car_type)
    driver.quit()

    if car_data:
        save_to_mysql(car_data, host, user, password, database)
        save_to_csv(car_data)  # Save to CSV
    else:
        print("No car data found.")

# Run Scraper
if __name__ == "__main__":
    main("https://www.cardekho.com/used-pickup-truck+cars+in+bangalore", "hyderabad", "pickup-truck", "localhost", "root", "root", "car_details")


Fetching data from: https://www.cardekho.com/used-pickup-truck+cars+in+bangalore
Scrolling completed.
2 records inserted into MySQL!
Data saved to car_details.csv


In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import mysql.connector
import time
import re
from itertools import product

# Function to Initialize WebDriver
def get_driver():
    options = Options()
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    options.add_argument("--headless")  # Run in headless mode
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Function to Extract Mileage from Car Details Page
def extract_mileage(driver, car_url):
    try:
        driver.get(car_url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "vdpCard-carFeatures"))
        )
        for _ in range(3):  # Retry mechanism
            try:
                mileage_element = driver.find_elements(By.XPATH, "//li/div/div/div[contains(text(), 'Mileage')]/following-sibling::span")
                if mileage_element:
                    return mileage_element[0].text.strip()
            except Exception as e:
                print(f"Retrying mileage extraction due to: {e}")
                time.sleep(2)
        return "N/A"
    except Exception as e:
        print(f"Error extracting mileage: {e}")
        return "N/A"

def extract_price(price_str):
    # Remove currency symbol and extract numeric parts
    match = re.search(r"([\d.]+)\s*Lakh", price_str)
    
    if match:
        # Convert Lakh to integer (1 Lakh = 100000)
        return int(float(match.group(1)) * 100000)
    
    match = re.search(r"([\d,]+)", price_str)
    if match:
        # Remove commas and convert to integer for direct prices
        return int(match.group(1).replace(",", ""))
    
    return None  # Return None if no valid price found
# Function to Extract Car Details
def extract_car_data(driver, location, car_type):
    car_list = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "ClsBox"))
    )

    while True:  # Keep retrying until we get all data
        try:
            car_elements = driver.find_elements(By.CLASS_NAME, "cardColumn")  # Refresh elements
            for index in range(len(car_elements)):
                try:
                    car_elements = driver.find_elements(By.CLASS_NAME, "cardColumn")  # Re-fetch elements
                    element = car_elements[index]  # Get element fresh

                    name_element = element.find_elements(By.XPATH, ".//h3[contains(@class, 'title')]")
                    name = name_element[0].text.strip() if name_element else "N/A"

                    link_element = element.find_elements(By.XPATH, ".//a[contains(@href, '/used-car-details/')]")
                    car_url = link_element[0].get_attribute("href") if link_element else None

                    price_element = element.find_elements(By.XPATH, ".//div[contains(@class, 'Price')]/p")
                    price = price_element[0].text.strip() if price_element else "N/A"

                    # Navigate to the car details page
                    if car_url:
                        driver.get(car_url)
                        mileage = extract_mileage(driver, car_url)
                        driver.back()  # Return to the listing page
                        WebDriverWait(driver, 5).until(
                            EC.presence_of_all_elements_located((By.CLASS_NAME, "cardColumn"))
                        )  # Wait for page to reload

                    else:
                        mileage = "N/A"

                    car_list.append({
                        "Name": name,
                        "Price": price,
                        "Mileage": mileage
                    })

                except Exception as e:
                    print(f"Retrying due to stale element: {e}")
                    time.sleep(1)  # Small delay before retrying

            break  # Exit loop when done

        except Exception as e:
            print(f"Error extracting data: {e}")
            time.sleep(2)  # Delay before retrying

    return car_list


    
# Function to Save Data into MySQL
def save_to_mysql(data, host, user, password, database):
    try:
        conn = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database
        )
        cursor = conn.cursor()

        # Insert Query
        query = """
        INSERT INTO used_cars (name, price, kilometers_driven, fuel_type, transmission_type, location, car_type, model, mileage)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        # Insert Each Record
        for car in data:
            values = (
                car["Name"],
                car["Price"],
                car["Kilometers_Driven"],
                car["Fuel_Type"],
                car["Transmission_Type"],
                car["Location"],
                car["Car_Type"],
                car["Model"],
                car["Mileage"]
            )
            cursor.execute(query, values)

        conn.commit()
        print(f"{len(data)} records inserted successfully into MySQL!")

    except mysql.connector.Error as err:
        print(f"MySQL Error: {err}")

    finally:
        cursor.close()
        conn.close()

# Main Function to Extract and Save Data
def main(url, location, car_type, host, user, password, database):
    print(f"Fetching data from: {url}")

    driver = get_driver()
    driver.get(url)
    time.sleep(2)  # Allow page to load

    # Extract Data
    car_data = extract_car_data(driver, location, car_type)

    # Close Driver
    driver.quit()

    if car_data:
        # Save Data into MySQL
        #save_to_mysql(car_data, host, user, password, database)
        print("Data found")
    else:
        print("No car data found.")

# Example Usage
if __name__ == "__main__":
    # MySQL Connection Details
    host = "localhost"
    user = "root"
    password = "root"
    database = "student"

    # Define Locations and Model Types
    locations = {"Hyderabad"}
    model_types = {"hatchback"}
    
    # Generate URLs dynamically
    base_url = "https://www.cardekho.com/used-{model_type}+cars+in+{location}"
    # https://www.cardekho.com/used-sedan+cars+in+hyderabad
    
    # Generate all combinations of model types and locations
    for model, location in product(model_types, locations):
        url = base_url.format(model_type=model.lower(), location=location.lower())
        print(url)
        main(url, location, model, host, user, password, database)


https://www.cardekho.com/used-hatchback+cars+in+hyderabad
Fetching data from: https://www.cardekho.com/used-hatchback+cars+in+hyderabad
Data found


In [27]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import mysql.connector
import time
import re
from itertools import product

# Function to Initialize WebDriver
def get_driver():
    options = Options()
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    options.add_argument("--headless")  # Run in headless mode
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Function to Extract Mileage from Car Details Page
def extract_mileage(driver, car_url):
    try:
        driver.get(car_url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "vdpCard-carFeatures"))
        )
        for _ in range(3):  # Retry mechanism
            try:
                mileage_element = driver.find_elements(By.XPATH, "//li/div/div/div[contains(text(), 'Mileage')]/following-sibling::span")
                if mileage_element:
                    return mileage_element[0].text.strip()
            except Exception as e:
                print(f"Retrying mileage extraction due to: {e}")
                time.sleep(2)
        return "N/A"
    except Exception as e:
        print(f"Error extracting mileage: {e}")
        return "N/A"

def extract_price(price_str):
    # Remove currency symbol and extract numeric parts
    match = re.search(r"([\d.]+)\s*Lakh", price_str)
    
    if match:
        # Convert Lakh to integer (1 Lakh = 100000)
        return int(float(match.group(1)) * 100000)
    
    match = re.search(r"([\d,]+)", price_str)
    if match:
        # Remove commas and convert to integer for direct prices
        return int(match.group(1).replace(",", ""))
    
    return None  # Return None if no valid price found
# Function to Extract Car Details
def extract_car_data(driver, location, car_type):
    car_list = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "ClsBox"))
    )

    while True:  # Keep retrying until we get all data
        try:
            car_elements = driver.find_elements(By.CLASS_NAME, "cardColumn")  # Refresh elements
            for index in range(len(car_elements)):
                try:
                    # name_element = element.find_elements(By.XPATH, ".//h3[contains(@class, 'title')]")
                    # name = name_element[0].text.strip() if name_element else "N/A"
                    car_elements = driver.find_elements(By.CLASS_NAME, "cardColumn")  # Re-fetch elements
                    element = car_elements[index]  # Get element fresh
                   
                    name_element = element.find_elements(By.XPATH, ".//h3[contains(@class, 'title')]")
              
                    name = name_element[0].text.strip() if name_element else "N/A"
    
                    link_element = element.find_elements(By.XPATH, ".//a[contains(@href, '/used-car-details/')]")
                    car_url = link_element[0].get_attribute("href") if link_element else None
                    
                    price_element = element.find_elements(By.XPATH, ".//div[contains(@class, 'Price')]/p")
                    price = price_element[0].text.strip() if price_element else "N/A"
                    price = extract_price(price)
                    
                    details_element = retry_on_stale_element(element, By.CLASS_NAME, "dotsDetails")
                    details_text = details_element[0].get_attribute("innerText").strip() if details_element else ""
                    details_list = [item.strip() for item in details_text.split("•")] if details_text else []
                    kms = details_list[0].replace(" kms", "").replace(",", "").strip() if len(details_list) > 0 else "0"
                    fuel_type = details_list[1].strip() if len(details_list) > 1 else "Unknown"
                    transmission_type = details_list[2].strip() if len(details_list) > 2 else "Unknown"
                    print(transmission_type)
                    try:
                        kms = int(kms)
                    except ValueError:
                        kms = 0
                    # location_element = retry_on_stale_element(element, By.XPATH, ".//div[contains(@class, 'distanceText')]")
                    # full_location = location_element[0].text.strip() if location_element else location
                    # full_location = full_location.split(",")[-1].strip()
                    model = name.split()[0]

                    # Navigate to the car details page
                    if car_url:
                        driver.get(car_url)
                        mileage = extract_mileage(driver, car_url)
                        driver.back()  # Return to the listing page
                        WebDriverWait(driver, 5).until(
                            EC.presence_of_all_elements_located((By.CLASS_NAME, "cardColumn"))
                        )  # Wait for page to reload

                    else:
                        mileage = "N/A"

                    car_list.append({
                    "Name": name,
                    "Price": price,
                    "Kilometers_Driven": kms,
                    "Fuel_Type": fuel_type,
                    "Transmission_Type": transmission_type,
                    "Location": location,
                    "Car_Type": car_type,
                    "Model": model,
                    "Mileage": mileage,
                    "url":car_url
                    })
                    print(car_list)
                except Exception as e:
                    print(f"Retrying due to stale element: {e}")
                    time.sleep(1)  # Small delay before retrying

            break  # Exit loop when done

        except Exception as e:
            print(f"Error extracting data: {e}")
            time.sleep(2)  # Delay before retrying

    return car_list


    
# Function to Save Data into MySQL
def save_to_mysql(data, host, user, password, database):
    try:
        conn = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database
        )
        cursor = conn.cursor()

        # Insert Query
        query = """
        INSERT INTO dummy_car (name, price, kilometers_driven, fuel_type, transmission_type, location, car_type, model, mileage)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        # Insert Each Record
        for car in data:
            values = (
                car["Name"],
                car["Price"],
                car["Kilometers_Driven"],
                car["Fuel_Type"],
                car["Transmission_Type"],
                car["Location"],
                car["Car_Type"],
                car["Model"],
                car["Mileage"]
            )
            cursor.execute(query, values)

        conn.commit()
        print(f"{len(data)} records inserted successfully into MySQL!")

    except mysql.connector.Error as err:
        print(f"MySQL Error: {err}")

    finally:
        cursor.close()
        conn.close()

# Main Function to Extract and Save Data
def main(url, location, car_type, host, user, password, database):
    print(f"Fetching data from: {url}")

    driver = get_driver()
    driver.get(url)
    time.sleep(2)  # Allow page to load

    # Extract Data
    car_data = extract_car_data(driver, location, car_type)

    # Close Driver
    driver.quit()

    if car_data:
        # Save Data into MySQL
        save_to_mysql(car_data, host, user, password, database)
        print("Data found")
    else:
        print("No car data found.")

# Example Usage
if __name__ == "__main__":
    # MySQL Connection Details
    host = "localhost"
    user = "root"
    password = "root"
    database = "car_details"

    # Define Locations and Model Types
    locations = {"Hyderabad"}
    model_types = {"hatchback"}
    
    # Generate URLs dynamically
    base_url = "https://www.cardekho.com/used-{model_type}+cars+in+{location}"
    # https://www.cardekho.com/used-sedan+cars+in+hyderabad
    
    # Generate all combinations of model types and locations
    for model, location in product(model_types, locations):
        url = base_url.format(model_type=model.lower(), location=location.lower())
        print(url)
        main(url, location, model, host, user, password, database)


https://www.cardekho.com/used-hatchback+cars+in+hyderabad
Fetching data from: https://www.cardekho.com/used-hatchback+cars+in+hyderabad
1111111111111
2222222222222222222
333333333333333333333333333
Manual
[{'Name': '2016 Maruti Swift VDI BSIV', 'Price': 530000, 'Kilometers_Driven': 81562, 'Fuel_Type': 'Diesel', 'Transmission_Type': 'Manual', 'Location': 'Hyderabad', 'Car_Type': 'hatchback', 'Model': '2016', 'Mileage': '25.2 kmpl', 'url': 'https://www.cardekho.com/used-car-details/used-Maruti-swift-vdi-bsiv-cars-Hyderabad_aaa62be8-16be-4dd0-856a-3655e6a6f9a7.htm?adId=725&adType=41'}]
1111111111111
2222222222222222222
333333333333333333333333333
Manual
[{'Name': '2016 Maruti Swift VDI BSIV', 'Price': 530000, 'Kilometers_Driven': 81562, 'Fuel_Type': 'Diesel', 'Transmission_Type': 'Manual', 'Location': 'Hyderabad', 'Car_Type': 'hatchback', 'Model': '2016', 'Mileage': '25.2 kmpl', 'url': 'https://www.cardekho.com/used-car-details/used-Maruti-swift-vdi-bsiv-cars-Hyderabad_aaa62be8-16be-4dd0-